In [1]:
##modules
#%matplotlib widget
%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5
# import pymer4 
# from pymer4.models import lmer, compare

import pickle

In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
        
    
with open(datadir / f"subjects_remove_{modality}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)
    



📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

# Tables and paths
Now, as specially in the event_preprocessing we are going to work with lots of different comparisons, we are going to create, instead of variables only, we are going to use comparisons 

Example: comparison_1= [zinnen, woorden]
so variable 1= comparison_1[0]

then we will create a for loop, to print analysis for all conditions

### Paths
ACW_path
PLE_path

### Tables and metric_individuals
#### Block

*ACW*


f"acw_results_subjects_all_{layer_script}.pickle" 
if you want the acf and the whole table   f"autocorrelation_subjects_all_{layer_script}.pickle"

f"autocorrelation_subjects_all_{layer_script}.pickle"

**variables** =  acw_50_elect_all_epoch_all //  'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all'


*PLE*
f"table_PLE_slope_intercept_subjects_all_{layer_script}.pickle"

f"table_PLE_subjects_all_{layer_script}.pickle"


#### dynamic

f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle"

f"dynamic_autocorrelation_subjects_all_{layer_script}.pickle"

**variables** = acw_50_slope_elect_all_epoch_all   // acw_50_std_elect_all_epoch_all 

In [3]:
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


In [37]:
## Variables of script

## Variables of script
##path=analysis_path
path = ACW_path
if filtering:
    name_table = f"acw_results_subjects_all_{filter_name}_{layer_script}.pickle" 
else:   
    name_table = f"acw_results_subjects_all_{layer_script}.pickle" 
table_df = pd.read_pickle(f"{path}\\{name_table}")

# Extraer sujetos del dataframe
subjects = table_df["Subject"].unique()

# Filtrar sujetos eliminados
subjects = [s for s in subjects if s not in subjects_remove]

# Filtrar tabla también
table_df = table_df.query("Subject in @subjects")

# Remove them also from table
#metric_individual NAMES
metrics= ['acw_50_elect_all_epoch_all',"acw_0_elect_all_epoch_all"]
metric_individual='acw_50_elect_all_epoch_all'


if '_elect_all_epoch_all' in metric_individual:
    print("yes")
    metric_individual_name = metric_individual.replace('_elect_all_epoch_all', "")
else:
    metric_individual_name = metric_individual
    
if filtering:
    name_table_fooof =f"df_fooof_subject_all_{filter_name}_fixed_{layer_script}.pickle" 
else:
    name_table_fooof =f"df_fooof_subject_all_fixed_{layer_script}.pickle"


table_fooof_df = pd.read_pickle(f"{path}\\{name_table_fooof}")
table_fooof_df = table_fooof_df.query("Subject in @subjects")



yes


In [ ]:
table_fooof_df.head()

#FOOOF table

freq_bands = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 40)
}    
    
##CONDITION NAMES
if layer_script == "event":
    zinnen_noQ = ["zinnen_RC_plus", "zinnen_RC_neg"]
    woorden_noQ = ["woorden_RC_plus", "woorden_RC_neg"]

    zinnen_RCplus = ["zinnen_RC_plus"]
    zinnen_RCneg = ["zinnen_RC_neg"]

    question_hit = [
        "zinnen_RC_neg_question_hit",
        "zinnen_RC_plus_question_hit",
        "woorden_RC_neg_question_hit",
        "woorden_RC_plus_question_hit"
    ]

    question_incorrect = [
        "zinnen_RC_neg_question_incorrect",
        "zinnen_RC_plus_question_incorrect",
        "woorden_RC_neg_question_incorrect",
        "woorden_RC_plus_question_incorrect"
    ]



if layer_script == "block":
    ## Comparisons
    comparison_1 = ["ZINNEN", "WOORDEN"]
    comparison_2=[]
    comparison_3=[]
    comparison_4=[]

    comparisons = [comparison_1, comparison_2,
                comparison_3, comparison_4]
elif layer_script == "event":
    comparison_1 = [zinnen_noQ, woorden_noQ]          # ZINNEN vs WOORDEN sin preguntas
    comparison_2 = [zinnen_RCplus, zinnen_RCneg]      # RC+ vs RC− en zinnen
    comparison_3 = [question_hit, question_incorrect] # Trials correctos vs incorrectos

    comparisons = [comparison_1, comparison_2, comparison_3]


#TYPE OF DIFFERENCES
difference = "normal"  # "normal" or "inverse"

## this will be used in permutation differences
if difference == "normal":
    alternative = "greater" 
    # this is for cluster permutation test
    tail=1 
    threshold_direction =1
else:
    alternative="less"
    tail=-1
    threshold_direction =-1


p_cluster_value = 0.05 # p-value for cluster permutation test

number_decimals=6
# ##creacion dataframe acw_0

# acw_50_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_50_elect_all_epoch_all"]]
# acw_50_df.to_pickle(ACW_path / f"acw_50_df.pickle")

# del acw_50_df
# acw_0_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "metric_individual_elect_all_epoch_all"]]
# acw_0_df.to_pickle(ACW_path / f"acw_0_df.pickle")

# del acw_all
# del acw_0_df

# channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
# channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

# channels_mag=channels_mag.tolist()
# print5(channels_mag)
# indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
# indice_channels_efectivos=indice_channels_efectivos.tolist()
# # del channels

##valores de las columnas
condition = table_df["Condition"].unique()
print5("Condiciones en los datos:", condition)

subjects = table_df["Subject"].unique()
print5("Sujetos en los datos:", subjects)

elect_all=  table_df["Elect"].unique()
print5("sensores en los datos:", elect_all)

epochs_all=  table_df["Epoch"].unique()
print5("Epochs en los datos:", epochs_all)
## get info 


#read epochs to build evoked
# leer epochs del sujeto (único archivo)
epochs = mne.read_epochs(epochs_clean_path / f"{subj}_epochs_{layer_script}-epo.fif")

if layer_script == "block":
# seleccionar ensayos de condición "zinnen"
    epochs_zinnen = epochs["fix_ZINNEN"]
if layer_script == "event":
    # seleccionar ensayos de condición "zinnen"
    epochs_zinnen = epochs["begin_zinnen_RC_neg"]

# obtener evoked
evoked_zinnen = epochs_zinnen.copy().pick("mag", exclude="bads").average()

info = evoked_zinnen.info

# limpieza memoria (opcional)
del epochs, epochs_zinnen




,Subject,Condition,Epoch,Elect,delta,theta,alpha,beta,gamma,offsets,exponents,r2,error
0,sub-V1001,WOORDEN,1,MLC12-4304,0.0,0.313377,0.321742,0.195510,0.0,-26.623734,1.093448,0.985273,0.038964
1,sub-V1001,WOORDEN,1,MLC13-4304,0.0,0.000000,0.340758,0.000000,0.0,-26.525484,1.091156,0.941521,0.092458
2,sub-V1001,WOORDEN,1,MLC14-4304,0.0,0.000000,0.415460,0.249342,0.0,-26.334741,1.272053,0.984859,0.047137
3,sub-V1001,WOORDEN,1,MLC15-4304,0.0,0.000000,0.415895,0.226756,0.0,-26.201337,1.297853,0.984957,0.046183
4,sub-V1001,WOORDEN,1,MLC16-4304,0.0,0.000000,0.450177,0.241165,0.0,-26.166647,1.238125,0.976374,0.059663


In [8]:
table_df


,Subject,Condition,Epoch,Elect,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all
0,sub-V1001,ZINNEN,0,MLC12-4304,0.020000,0.243333
1,sub-V1001,ZINNEN,0,MLC13-4304,0.023333,0.253333
2,sub-V1001,ZINNEN,0,MLC14-4304,0.026667,0.246667
3,sub-V1001,ZINNEN,0,MLC15-4304,0.030000,0.220000
4,sub-V1001,ZINNEN,0,MLC16-4304,0.026667,0.170000
...,...,...,...,...,...,...
1221475,sub-V1117,ZINNEN,42,MZF03-4304,0.013333,0.060000
1221476,sub-V1117,ZINNEN,42,MZO01-4304,0.013333,0.036667
1221477,sub-V1117,ZINNEN,42,MZO02-4304,0.013333,0.033333
1221478,sub-V1117,ZINNEN,42,MZO03-4304,0.013333,0.040000


In [9]:
table_fooof_df


,Subject,Condition,Epoch,Elect,delta,theta,alpha,beta,gamma,offsets,exponents,r2,error
0,sub-V1001,WOORDEN,1,MLC12-4304,0.0,0.313377,0.321742,0.195510,0.0,-26.623734,1.093448,0.985273,0.038964
1,sub-V1001,WOORDEN,1,MLC13-4304,0.0,0.000000,0.340758,0.000000,0.0,-26.525484,1.091156,0.941521,0.092458
2,sub-V1001,WOORDEN,1,MLC14-4304,0.0,0.000000,0.415460,0.249342,0.0,-26.334741,1.272053,0.984859,0.047137
3,sub-V1001,WOORDEN,1,MLC15-4304,0.0,0.000000,0.415895,0.226756,0.0,-26.201337,1.297853,0.984957,0.046183
4,sub-V1001,WOORDEN,1,MLC16-4304,0.0,0.000000,0.450177,0.241165,0.0,-26.166647,1.238125,0.976374,0.059663
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1161535,sub-V1117,ZINNEN,42,MZF03-4304,0.0,0.285270,0.412647,0.215224,0.0,-27.123500,0.815780,0.984401,0.034952
1161536,sub-V1117,ZINNEN,42,MZO01-4304,0.0,0.000000,0.712440,0.643537,0.0,-26.498561,0.924303,0.977973,0.046381
1161537,sub-V1117,ZINNEN,42,MZO02-4304,0.0,0.000000,0.710358,0.499493,0.0,-26.644272,0.863191,0.983175,0.040097
1161538,sub-V1117,ZINNEN,42,MZO03-4304,0.0,0.000000,0.517961,0.371871,0.0,-26.994866,0.876358,0.984981,0.037204


## Selection of values of ACW

In [ ]:
### testing with pandas, nomenclature includes ONLY what is preserved, not what has been averaged

# this is for comparing between conditions, as it has dimension=number of subjects
metric_individual_cond_subj_epoch_mean_elect_mean= table_df.groupby(["Subject", "Condition"])[metric_individual].mean().reset_index()

#this is for plotting results (plot_topomap) as it has dimension=number of electrodes
metric_individual_cond_subj_mean_epoch_mean_elect=table_df.groupby(["Condition", "Elect"])[metric_individual].mean().reset_index()

#this is for cluster permutation test, as it has dimension=number of subjects and electrodes 
metric_individual_cond_subj_epoch_mean_elect=table_df.groupby(["Subject", "Condition", "Elect"])[metric_individual].mean().reset_index()

## Selection of values of Bands

In [ ]:
#as i´m not interested in the actual epoch numbers, 
# just in matching them between both dataframes, I RENAME THE NUMBER in epoch columns
def renumber_epochs(df):
    df = df.copy()
    df["Epoch"] = (
        df.groupby(["Subject", "Condition"])["Epoch"]
        .transform(lambda x: pd.factorize(x)[0])
    )
    return df




table_df_r = renumber_epochs(table_df)
table_fooof_df_r = renumber_epochs(table_fooof_df)

In [29]:
## this code checks differences in unique values of columns between both dataframes

cols = ["Subject", "Condition", "Epoch", "Elect"]

# for col in cols:
#     print(f"\n--- {col} ---")
#     print("table_df:")
#     print(table_df[col].unique())
#     print("table_fooof_df:")
#     print(table_fooof_df[col].unique())
for col in cols:
    set_df = set(table_df_r[col].unique())
    set_foo = set(table_fooof_df_r[col].unique())

    print(f"\n--- {col} ---")
    print("Solo en table_df:", set_df - set_foo)
    print("Solo en table_fooof_df:", set_foo - set_df)


--- Subject ---
Solo en table_df: set()
Solo en table_fooof_df: set()

--- Condition ---
Solo en table_df: set()
Solo en table_fooof_df: set()

--- Epoch ---
Solo en table_df: set()
Solo en table_fooof_df: set()

--- Elect ---
Solo en table_df: set()
Solo en table_fooof_df: set()


In [ ]:
table_merged_df = pd.merge(
    table_df_r,
    table_fooof_df_r,
    on=["Subject", "Condition", "Epoch", "Elect"],
    how="outer",
    indicator=True
)
## this code checks that there is no differences between both dataframes in the common columns

print(table_merged_df["_merge"].value_counts())

In [35]:
filter_name

'filt_1-40'

In [34]:
if filtering:
    output_path = ACW_path / f"table_merged_df_{filter_name}_{layer_script}.csv"
else:
    output_path = ACW_path / f"table_merged_df_{layer_script}.csv"

table_merged_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to g:\MOUS_204\MOUS_visual\output_analysis\analysis_block\acw_block\table_merged_df_filt_1-40_block.csv


In [39]:
metric_individuals_with_fooof = [metric_individual] + list(freq_bands.keys())
metric_individuals_with_fooof

['acw_50_elect_all_epoch_all', 'delta', 'theta', 'alpha', 'beta', 'gamma']

In [ ]:
## this is the unon of metric_individual and other dataframes

metric_individuals_with_fooof = [metric_individual] + list(freq_bands.keys())

# this is for comparing between conditions, as it has dimension=number of subjects
table_df_merged_cond_subj_epoch_mean_elect_mean= table_merged_df.groupby(["Subject", "Condition"])[metric_individuals_with_fooof].mean().reset_index()

#this is for plotting results (plot_topomap) as it has dimension=number of electrodes
table_df_merged_cond_subj_mean_epoch_mean_elect=table_merged_df.groupby(["Condition", "Elect"])[metric_individuals_with_fooof].mean().reset_index()

#this is for cluster permutation test, as it has dimension=number of subjects and electrodes 
table_df_merged__cond_subj_epoch_mean_elect=table_merged_df.groupby(["Subject", "Condition", "Elect"])[metric_individuals_with_fooof].mean().reset_index()

# Plots 

## Plot topomat of activation

In [ ]:

for comparison in comparisons:
    for exp_condition in comparison:
        # Filtrar el DataFrame por la condición actual
        if layer_script == "block":
            filtered_df = metric_individual_cond_subj_mean_epoch_mean_elect.query("Condition == @exp_condition")
        elif layer_script == "event":
            filtered_df = metric_individual_cond_subj_mean_epoch_mean_elect.query("Condition in @exp_condition")
            # ✅ Promediar entre condiciones del grupo
            filtered_df = (
                filtered_df
                .groupby("Elect")[metric_individual]
                .mean()
                .reset_index()
            )
            

        # Extraer los valores de la métrica y los nombres de los electrodos
        values = filtered_df[metric_individual].values.flatten()
        electrodes = filtered_df["Elect"].values

        # Asegurarse de que los valores y los electrodos tienen la misma longitud
        if len(values) != len(electrodes):
            raise ValueError("Los valores y los electrodos deben tener la misma longitud.")

        # Crea figura + eje manual
        fig, ax = plt.subplots(figsize=(6, 6))

        # Imprimir para depuración
        # print5(f"Valores para {exp_condition}: {values}")
        # print5(f"Electrodos para {exp_condition}: {electrodes}")
        
        
        # Dibujar topomap DENTRO de ese eje
        im, _ = mne.viz.plot_topomap(
            data=values,
            pos=info,
            axes=ax,           # ← esto asegura que se dibuja donde tú quieres
            cmap='RdBu_r',
            vlim=(-np.max(np.abs(values)), np.max(np.abs(values))),
            mask=None,
            contours=0,
            show=False         # ← evita que se muestre automáticamente
        )

        # Título
        fig.suptitle(f"{metric_individual_name} values in {exp_condition}", fontsize=20)

        # Añadir barra de color (leyenda)
        cbar = fig.colorbar(im, ax=ax, shrink=0.6)
        cbar.set_label(f"{metric_individual_name} value")

        # Mostrar figura
        plt.show()


In [ ]:
comparison

### linear plots

In [ ]:
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib.pyplot as plt

for comparison in comparisons:
    if not comparison:
        continue

    # Crear diccionario con arrays planos por grupo/condición
    dict_conditions_flat = {}

    for exp_condition in comparison:

        # Si la condición es una lista -> agrupar
        if isinstance(exp_condition, list):
            condition_flat = (
                metric_individual_cond_subj_epoch_mean_elect_mean
                .query("Condition in @exp_condition")[metric_individual]
                .values.flatten()
            )
            label = exp_condition[0].split("_")[0] + "_AVG"
            dict_conditions_flat[label] = condition_flat
        
        # Si es una condición individual
        else:
            condition_flat = (
                metric_individual_cond_subj_epoch_mean_elect_mean
                .query("Condition == @exp_condition")[metric_individual]
                .values.flatten()
            )
            dict_conditions_flat[exp_condition] = condition_flat

    # ---- PLOT ----
    plt.figure(figsize=(10, 5))
    colors = ['blue', 'red', 'green', 'orange', 'purple']

    for idx, (label, data) in enumerate(dict_conditions_flat.items()):

        counts, bins, _ = plt.hist(
            data,
            bins=150,
            alpha=0.4,
            label=f'{label}',
            color=colors[idx % len(colors)]
        )

        kde = gaussian_kde(data)
        x_vals = np.linspace(data.min(), data.max(), 300)
        scaled_kde = kde(x_vals) * len(data) * np.diff(bins)[0]

        plt.plot(
            x_vals,
            scaled_kde,
            linestyle='--',
            color=colors[idx % len(colors)],
            label=f'{label} (KDE)'
        )

    plt.xlabel(f"{metric_individual_name}(s)")
    plt.ylabel("Número de valores")
    plt.title(f"Histograms + KDE for {metric_individual_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# General differences between woorden and zinnen conditions

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
table_df_merged_cond_subj_epoch_mean_elect_mean[metric_individuals] = scaler.fit_transform(table_df_merged_cond_subj_epoch_mean_elect_mean[metric_individuals])

metric_individuals_with_fooof = [metric_individual] + list(freq_bands.keys())

this is for comparing between conditions, as it has dimension=number of subjects
table_df_merged_cond_subj_epoch_mean_elect_mean= table_merged_df.groupby(["Subject", "Condition"])[metric_individuals_with_fooof].mean().reset_index()

this is for plotting results (plot_topomap) as it has dimension=number of electrodestable_df_merged_cond_subj_mean_epoch_mean_elect=table_merged_df.groupby(["Condition", "Elect"])[metric_individuals_with_fooof].mean().reset_index()

this is for cluster permutation test, as it has dimension=number of subjects and electrodes 
table_df_merged__cond_subj_epoch_mean_elect=table_merged_df.groupby(["Subject", "Condition", "Elect"])[metric_individuals_with_fooof].mean().reset_index()

In [ ]:
import Pymer4 
from pymer4.models import lmer, compare
#metric_individual_cond_subj_epoch_mean_elect shouldnt be used but plot gets uninformative, when all subjects, use metric_individual_cond_subj_epoch_mean_elect_mean
for comparison in comparisons:
    if comparison:
        print(comparison)

        formula = f"{metric_individual} ~ Condition + " + " + ".join(list(freq_bands.keys()))
        # formula=f"{metric_individual} ~ Condition"
        modelo = smf.mixedlm(
            formula,
            data=table_merged_df,
            groups="Subject",
            vc_formula={
                "Elect": "0 + C(Elect)",
                "Epoch": "0+C(Epoch)"}
        ).fit()

        print(modelo.summary())
        




In [ ]:
table_df_merged_cond_subj_epoch_mean_elect_mean.shape

# Cluster analysis

## Datos a comparar

ahora tengo por cada condición: subject x epoch x channel

#### procedimiento

- promediar a nivel de epoca para tener subject x channel (luego probaré a hacerlo de otra manera sin promediar)

- ejecutar permutation cluster based analysis usando mi acw_zinnen vs mi acw_woorden  que va a estar como subject x channel

    - el cluster solo puede ser espacial, porque mis epocas son discontinuas, creo que esto es adjacency 

#### Resultados

T_obs: La estadística observada en cada punto.
clusters: Los clústeres formados por puntos significativos adyacentes.
cluster_p_values: Los valores p de cada clúster (ya corregidos por comparaciones múltiples).
H0: La distribución nula obtenida por permutaciones.



Supongo que clusters y sus valores de significacioón
Luego no se como se compara entre condiciones, pero de momemento vamos así

## Cluster in differences

In [ ]:
# ##get adjacency value
# adjacency_reduced =  pd.read_pickle(channels_structure_path /f"adjacency_reduced_{modality}.pkl")
# ###print5(f"shape adjacency_reduced: {adjacency_reduced.shape}")

In [ ]:
# ##note that condition_1 and condition_2 are 2D arrays with shape (n_subjects, n_channels)

# #here im using various things
# ##.query("Condition ==@cond") = [metric_individual_cond_subj_epoch_mean_elect["Condition"]=cond]
# #pivot index="Subject",columns="Elect", values=metric_individual) gives toy a a table with indexes and columns
# #to_numpy converts the table to a numpy array, removin names of indexes and columns

# for cond in condition_names:
#     if cond==condition_1_name:
#         print(f"x is {cond} and {condition_1_name}")
#         x= metric_individual_cond_subj_epoch_mean_elect.query("Condition ==@cond").pivot(index="Subject",columns="Elect", values=metric_individual).to_numpy()   
#     if cond==condition_2_name:
#         print(f"y is {cond} and {condition_2_name}")
#         y=metric_individual_cond_subj_epoch_mean_elect.query("Condition ==@cond").pivot(index="Subject",columns="Elect", values=metric_individual).to_numpy()   

# print(f"x.shape is {x.shape}")
# print(f"y.shape is {y.shape}")
# diff= x-y 
# print5(f"diff.shape is {diff.shape}")




# # as I´m going to apply differences to same subjects, i need a DEPENDENT t test kind of thing, 
# # so instead of mne.stats.permutation_cluster_test I´m going to use mne.stats.permutation_cluster_1samp_test

# # mne.stats.permutation_cluster_1samp_test(X, threshold=None, n_permutations=1024, tail=0, stat_fun=None, adjacency=None, n_jobs=None, seed=None, max_step=1, exclude=None, 
# #                                          step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)

# # as im going to put zinnen before woorden i need tail=1

# #instead of puting X with bot condigions, i will put the difference between them diff=zinnen-woorden
# # i modify the value of the tail to be -1, because i want to see if the values of zinnen are lower than the values of woorden
# from scipy.stats import t

# t_thresh = (threshold_direction ) * t.ppf(1 - p_cluster_value, df=len(subjects) - 1)


# t_obs_diff, clusters_diff, clusters_pv_diff, H0=mne.stats.permutation_cluster_1samp_test(diff, threshold=t_thresh, n_permutations=1024, tail=tail, 
#                                                                           stat_fun=None, adjacency=adjacency_reduced, n_jobs=15, seed=None, max_step=1, exclude=None,
#                                                                           step_down_p=0, t_power=1, out_type='indices', check_disjoint=False, buffer_size=1000, verbose=None)


# # Obtenemos los clusters como listas de índices
# # Filtrar clusters con p < 0.05
# #zip links clusters y p-values
# significant_clusters_diff = [
#     cluster for cluster, p in zip(clusters_diff, clusters_pv_diff) if p < 0.05
# ]

# print5(f"{len(significant_clusters_diff)} significative clusters found.")
# print5(f"{metric_individual_name}Clusters significative: {significant_clusters_diff}")

In [ ]:
# ###PLOTSS

# mask_diff = np.zeros(t_obs_diff.shape, dtype=bool)

# ##significant_clusters_diff is a list of tuples of arrays
# # cluster[0] is the electrodes of the  cluster
# for cluster in significant_clusters_diff:
#     mask_diff[cluster[0]] = True 



# ##in data i use the value of t_obs_diff
# #pos takes the information from evoked.info object
# #mask takes the mask_diff object
# #in t stat i don´t change it becausse the p value will be calculated with permutations, so no problem when using a parametric_individual test,
# # even if the distribution is non parametric_individual 

# # Crear figura y eje
# fig, ax = plt.subplots(figsize=(6, 6))

# # Dibujar el topomap
# im, _ = mne.viz.plot_topomap(
#     data=t_obs_diff,
#     pos=info,
#     mask=mask_diff,
#     axes=ax,
#     cmap='RdBu_r',
#     vlim=(-np.max(np.abs(t_obs_diff)), np.max(np.abs(t_obs_diff))),
#     mask_params=dict(marker='o', markerfacecolor='yellow', markersize=7),
#     contours=0,
#     show=False
# )

# # Añadir título
# fig.suptitle(f"{metric_individual_name} Clusters in differences, p_threshold={p_cluster_value}", fontsize=14)

# # Añadir barra de color
# cbar = fig.colorbar(im, ax=ax, shrink=0.6)
# cbar.set_label("T-statistic value")

# # Mostrar el gráfico
# plt.show()